# B3b Defense – 03 Feature Engineering

**Objective:** Build `defense_feature_matrix.csv` from `macro_features_defense.csv`.
Target variable: FDEFX (absolute level, USD millions). Features include:
- Lag features on ADEFNO absolute and ADEFNO_diff (lags 1–24, empirically motivated by Defense Capital Goods order-to-shipment lead times: ~3–6 months for small arms/ammunition, ~6–12 months for communications equipment, ~12–24 months for aircraft and missiles)
- Lag features on IPB52300S_diff (lags 1–12)
- Autoregressive lags on FDEFX absolute and FDEFX_diff (lags 1–6)
- Rolling window statistics on FDEFX (3m, 6m, 12m mean/std)
- Calendar features (month, quarter, year, is_q4)

In [1]:
import os
import pandas as pd
import numpy as np

In [2]:
DATA_PROCESSED = '../data/processed/'

# Load macro_features_defense.csv — explicit to_datetime() instead of parse_dates
# to avoid the slow dateutil fallback path in pandas 2.x for YYYY-MM strings.
input_path = os.path.join(DATA_PROCESSED, 'macro_features_defense.csv')
df = pd.read_csv(input_path)
df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True)
df.sort_index(inplace=True)

print(f'Loaded: {input_path}')
print(f'Shape: {df.shape}')
print(f'Date range: {df.index.min().date()} to {df.index.max().date()}')
df.head()

Loaded: ../data/processed/macro_features_defense.csv
Shape: (311, 6)
Date range: 2000-02-01 to 2025-12-01


,ADEFNO,ADEFNO_diff,IPB52300S,IPB52300S_diff,FDEFX,FDEFX_diff
date,,,,,,
2000-02-01,5162,-1997.0,69.3458,-1.6863,383.028,0.000
2000-03-01,3773,-1389.0,68.6844,-0.6614,383.028,0.000
2000-04-01,3568,-205.0,67.3689,-1.3155,399.592,16.564
2000-05-01,5023,1455.0,66.5616,-0.8073,399.592,0.000
2000-06-01,25893,20870.0,67.1028,0.5412,399.592,0.000


In [3]:
# Target: FDEFX absolute (not differenced).
# Rationale: FDEFX measures realized federal defense expenditures, which is semantically
# consistent with B1/B2 invoice-based forecasts in SAC. Absolute values preserve
# interpretability — forecast outputs can be read directly in USD millions without
# cumulative reconstruction from diffs.
target = df['FDEFX'].copy()

# Initialize feature DataFrame starting from the raw data
feat = df.copy()

In [4]:
# ADEFNO as leading indicator for FDEFX.
# Lag window 1-24 months empirically motivated by the heterogeneous order-to-shipment
# lead times in Defense Capital Goods: ~3-6 months (small arms, ammunition),
# ~6-12 months (communications equipment), ~12-24 months (aircraft, missiles).
# XGBoost will identify the dominant conversion window via feature importance.
for lag in range(1, 25):
    feat[f'ADEFNO_lag_{lag}'] = feat['ADEFNO'].shift(lag)
    feat[f'ADEFNO_diff_lag_{lag}'] = feat['ADEFNO_diff'].shift(lag)

print('ADEFNO absolute and diff lags 1–24 created.')

ADEFNO absolute and diff lags 1–24 created.


In [5]:
# Lag features for IPB52300S_diff (lags 1–12)
# Extended to 12 months (from 6) to allow XGBoost to capture seasonal patterns
# in defense production capacity across a full annual cycle.
for lag in range(1, 13):
    feat[f'IPB52300S_diff_lag_{lag}'] = feat['IPB52300S_diff'].shift(lag)

print('IPB52300S_diff lags 1–12 created.')

IPB52300S_diff lags 1–12 created.


In [6]:
# Autoregressive lags on FDEFX (target) — both absolute and differenced.
# AR lags up to 6 months capture short-term persistence in expenditure patterns;
# longer-range dynamics are delegated to the ADEFNO leading features.
for lag in range(1, 7):
    feat[f'fdefx_lag_{lag}'] = feat['FDEFX'].shift(lag)
    feat[f'FDEFX_diff_lag_{lag}'] = feat['FDEFX_diff'].shift(lag)

print('FDEFX absolute and diff AR lags 1–6 created.')

FDEFX absolute and diff AR lags 1–6 created.


In [7]:
# Rolling window features on FDEFX absolute level
# shift(1) is applied before rolling to prevent data leakage (no look-ahead).
# 12m rolling added to capture longer-term expenditure baselines.
fdefx_shifted = feat['FDEFX'].shift(1)

feat['fdefx_rolling_3m_mean']  = fdefx_shifted.rolling(window=3).mean()
feat['fdefx_rolling_3m_std']   = fdefx_shifted.rolling(window=3).std()
feat['fdefx_rolling_6m_mean']  = fdefx_shifted.rolling(window=6).mean()
feat['fdefx_rolling_6m_std']   = fdefx_shifted.rolling(window=6).std()
feat['fdefx_rolling_12m_mean'] = fdefx_shifted.rolling(window=12).mean()
feat['fdefx_rolling_12m_std']  = fdefx_shifted.rolling(window=12).std()

print('Rolling window features (3m, 6m, 12m mean/std) on FDEFX created with shift=1 to avoid leakage.')

Rolling window features (3m, 6m, 12m mean/std) on FDEFX created with shift=1 to avoid leakage.


In [8]:
# Calendar features derived from the DatetimeIndex
feat['month']   = feat.index.month
feat['quarter'] = feat.index.quarter
feat['year']    = feat.index.year
feat['is_q4']   = (feat['quarter'] == 4).astype(int)

print('Calendar features created: month, quarter, year, is_q4.')

Calendar features created: month, quarter, year, is_q4.


In [9]:
# Drop NaN rows introduced by lag and rolling operations
shape_before = feat.shape
feat.dropna(inplace=True)
shape_after = feat.shape

print(f'Shape before dropna: {shape_before}')
print(f'Shape after  dropna: {shape_after}')
print(f'Rows dropped: {shape_before[0] - shape_after[0]}')

Shape before dropna: (311, 88)
Shape after  dropna: (287, 88)
Rows dropped: 24


In [10]:
# Save defense_feature_matrix.csv to DATA_PROCESSED
out_path = os.path.join(DATA_PROCESSED, 'defense_feature_matrix.csv')
feat.to_csv(out_path)
print(f'Saved: {out_path}')
print()
print('head(3):')
print(feat.head(3).to_string())
print()
print('dtypes:')
print(feat.dtypes)

Saved: ../data/processed/defense_feature_matrix.csv

head(3):
            ADEFNO  ADEFNO_diff  IPB52300S  IPB52300S_diff    FDEFX  FDEFX_diff  ADEFNO_lag_1  ADEFNO_diff_lag_1  ADEFNO_lag_2  ADEFNO_diff_lag_2  ADEFNO_lag_3  ADEFNO_diff_lag_3  ADEFNO_lag_4  ADEFNO_diff_lag_4  ADEFNO_lag_5  ADEFNO_diff_lag_5  ADEFNO_lag_6  ADEFNO_diff_lag_6  ADEFNO_lag_7  ADEFNO_diff_lag_7  ADEFNO_lag_8  ADEFNO_diff_lag_8  ADEFNO_lag_9  ADEFNO_diff_lag_9  ADEFNO_lag_10  ADEFNO_diff_lag_10  ADEFNO_lag_11  ADEFNO_diff_lag_11  ADEFNO_lag_12  ADEFNO_diff_lag_12  ADEFNO_lag_13  ADEFNO_diff_lag_13  ADEFNO_lag_14  ADEFNO_diff_lag_14  ADEFNO_lag_15  ADEFNO_diff_lag_15  ADEFNO_lag_16  ADEFNO_diff_lag_16  ADEFNO_lag_17  ADEFNO_diff_lag_17  ADEFNO_lag_18  ADEFNO_diff_lag_18  ADEFNO_lag_19  ADEFNO_diff_lag_19  ADEFNO_lag_20  ADEFNO_diff_lag_20  ADEFNO_lag_21  ADEFNO_diff_lag_21  ADEFNO_lag_22  ADEFNO_diff_lag_22  ADEFNO_lag_23  ADEFNO_diff_lag_23  ADEFNO_lag_24  ADEFNO_diff_lag_24  IPB52300S_diff_lag_1  IPB52300S_dif